# Building the Quality Control (QC) Dashboard — Hands-on Practice

This notebook is a **long-form, workshop-style** course. You move from a minimal QC prototype to batch-scale checks, reviewer workflows, monitoring, and many independent pattern examples.

---

## Course map (all parts and sections)

Run sections **top to bottom** the first time so variables (`qc_df`, `batch_df`, `base_img`, `issue_matrix`, …) stay consistent.

### Part I — Core prototype (Steps 0–9)

| Step | Topic |
|------|--------|
| **0** | Dependencies (`pip` reminder; optional install). |
| **1** | Imports and output folders: `qc_dashboard_outputs/images`, `qc_dashboard_outputs/reports`. |
| **2** | Sample **AI-extracted** table with intentional defects. |
| **3** | **QC rules**, boolean issue matrix, **red highlighting** (`Styler`). |
| **4** | **Automated reports**: CSV inputs/issues/summary + **HTML** QC report. |
| **5** | Mock **high-resolution drawing**, synthetic **detections**. |
| **6** | **Simple UI** (`ipywidgets`): pick detection → **OpenCV overlay** preview. |
| **7** | **Human-in-the-loop**: edit fields, re-run checks, track review status. |
| **8** | **Export** corrected data + **audit** JSON; explain traceability. |
| **9** | Suggested **follow-on exercises** for the core track. |

### Part II — Extended practice track (batch QC, ops, capstone)

Introduced by **“Extended Practice Track (Long-Form Course)”**.

| Section | Topic |
|---------|--------|
| *(first code cell after the intro)* | **Severity weights**, row **priority score**, review bands (`pass` / `low` / `medium` / `high`). *(No separate “Step 10” heading in the notebook.)* |
| **Step 11** | Larger **synthetic batch** (stress-test rules). |
| **Step 12** | **KPI charts**: issue rate by field, pass/issue split, issues by component. |
| *(next code cell)* | **Reviewer queue**: priority score, **SLA buckets** (`none`, `this_week`, `today`, `urgent`). *(No separate “Step 13” heading.)* |
| **Step 14** | **Multi-reviewer** round-robin assignment + workload summary. |
| **Step 15** | Simulated **human corrections**, before/after metrics. |
| *(next code cell)* | **Retraining-style export**: AI vs human columns for changed fields → `qc_retraining_samples.csv`. *(No separate “Step 16” heading.)* |
| **Step 17** | **Batch PNG overlays** for reviewer packs (`images/batch_overlays/`). |
| **Step 18** | **Unit tests** (`assert`) for QC helper logic. |
| **Step 19** | **Monthly trend** simulation (issue rate vs SLA hit rate). |
| **Step 20** | **Capstone checklist** + closing note. |

### Part III — Additional worked examples (Examples 21–40)

Section: **“Additional Worked Examples (Extended Practice)”**. Short, runnable patterns:

- **21** — Component-specific pressure limits  
- **22** — Drawing number **regex** QC  
- **23** — **Duplicate** keys demo  
- **24** — **Cross-field** rule (e.g. Tank + high pressure)  
- **25** — **Text normalization**  
- **26** — **IQR outliers** per component  
- **27** — Confidence **buckets** vs issue rate  
- **28** — **Stratified** audit sampling  
- **29** — Simple **drift** comparison (two batches)  
- **30** — Detection JSON → **class colors + legend** overlay  
- **31** — Append-only **JSON Lines** event log  
- **32** — Simulated **AI vs human** crosstab  
- **33** — Export **`qc_rule_config.json`**  
- **34** — Per-row **`qc_reasons`** (explainability)  
- **35** — **Rolling** issue rate over simulated ingest times  
- **36** — **Auto-approval threshold** sweep vs issue rate  
- **37** — **BBox sanity** checks (vision QC)  
- **38** — Extra **styled HTML** batch preview report  
- **39** — **Pareto** chart of issues by field  
- **40** — Self-study **checklist**

---

## Learning outcomes (full scope)

After completing this notebook you should be able to:

- Define and encode **rule-based QC** on tabular extraction outputs; visualize failures clearly.
- Produce **repeatable artifacts** (CSV, HTML, JSON, PNG overlays) suitable for sharing and audits.
- Combine **vision overlays** with structured QC for engineering drawings.
- Operate a simple **HITL** loop and measure **before/after** quality.
- Scale thinking to **batches**, **queues**, **SLAs**, **reviewer assignment**, and **monitoring** plots.
- Reuse **Worked Examples 21–40** as building blocks for real pipelines (YOLO/OCR outputs, plant-specific limits).

---

## How to use this notebook

1. Run **Part I** once end-to-end to build intuition.  
2. Run **Part II** when teaching a longer session or simulating production load.  
3. Pick items from **Part III** à la carte for drills, homework, or code reviews.  
4. Keep outputs under `qc_dashboard_outputs/` as a **class portfolio**.


In [ ]:
# Step 0 - Install dependencies (run if needed)
# If your environment already has these packages, you can skip this cell.

# !pip install pandas numpy opencv-python ipywidgets matplotlib

print('Dependency cell ready. Uncomment pip command if packages are missing.')

In [ ]:
# Step 1 - Import libraries and configure paths

from pathlib import Path
import json
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

BASE_DIR = Path.cwd() / 'qc_dashboard_outputs'
IMG_DIR = BASE_DIR / 'images'
REPORT_DIR = BASE_DIR / 'reports'

for p in [BASE_DIR, IMG_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Output directory:', BASE_DIR)
print('Image directory :', IMG_DIR)
print('Report directory:', REPORT_DIR)

## Step 2 - Create Sample AI-Extracted Data

In a real project, this table comes from OCR or object detection + parsing.

For training purposes, we generate sample rows with intentional issues:

- Negative pressure values
- Temperatures outside engineering range
- Invalid confidence values
- Missing drawing references

These issues will be detected and highlighted by QC rules in the next step.

In [ ]:
# Build sample extraction table

data = [
    {'item_id': 'V-101', 'component_type': 'Valve', 'pressure_bar': 8.5,  'temperature_c': 65,  'confidence': 0.94, 'drawing_no': 'DWG-001'},
    {'item_id': 'P-202', 'component_type': 'Pump',  'pressure_bar': -1.2, 'temperature_c': 40,  'confidence': 0.88, 'drawing_no': 'DWG-001'},
    {'item_id': 'T-303', 'component_type': 'Tank',  'pressure_bar': 2.3,  'temperature_c': 420, 'confidence': 0.91, 'drawing_no': 'DWG-002'},
    {'item_id': 'F-404', 'component_type': 'Filter','pressure_bar': 1.1,  'temperature_c': 32,  'confidence': 1.20, 'drawing_no': 'DWG-003'},
    {'item_id': 'M-505', 'component_type': 'Motor', 'pressure_bar': 0.5,  'temperature_c': 22,  'confidence': 0.77, 'drawing_no': ''},
]

qc_df = pd.DataFrame(data)
qc_df

## Step 3 - Define QC Rules and Highlight Inconsistent Values in Red

We define simple validation rules:

- `pressure_bar` must be between `0` and `300`
- `temperature_c` must be between `-40` and `350`
- `confidence` must be between `0` and `1`
- `drawing_no` must not be empty

Then we create two outputs:

1. A boolean issue matrix (True = invalid).
2. A styled table where invalid values are colored red.

In [ ]:
def build_issue_matrix(df: pd.DataFrame) -> pd.DataFrame:
    issues = pd.DataFrame(index=df.index)
    issues['pressure_bar'] = ~df['pressure_bar'].between(0, 300)
    issues['temperature_c'] = ~df['temperature_c'].between(-40, 350)
    issues['confidence'] = ~df['confidence'].between(0, 1)
    issues['drawing_no'] = df['drawing_no'].astype(str).str.strip().eq('')
    return issues

issue_matrix = build_issue_matrix(qc_df)
issue_matrix

In [ ]:
def style_invalid_values(df: pd.DataFrame, issues: pd.DataFrame):
    # Red text + light red background for invalid fields
    styles = pd.DataFrame('', index=df.index, columns=df.columns)
    for col in issues.columns:
        styles.loc[issues[col], col] = 'color: #b00020; background-color: #ffe5e5; font-weight: bold;'
    return styles

styled_qc = qc_df.style.apply(lambda _: style_invalid_values(qc_df, issue_matrix), axis=None)

print('Rows with at least one issue:', issue_matrix.any(axis=1).sum())
display(styled_qc)

## Step 4 - Automated Report Generation

This step exports:

- `qc_input_data.csv`: raw extracted table
- `qc_issue_matrix.csv`: per-field QC flags
- `qc_summary.csv`: issue count by field
- `qc_report.html`: styled table for easy sharing

This is a minimal version of an automated QC report pipeline.

In [ ]:
# Save machine-readable reports
input_csv = REPORT_DIR / 'qc_input_data.csv'
issues_csv = REPORT_DIR / 'qc_issue_matrix.csv'
summary_csv = REPORT_DIR / 'qc_summary.csv'
html_report = REPORT_DIR / 'qc_report.html'

qc_df.to_csv(input_csv, index=False)
issue_matrix.to_csv(issues_csv, index=False)

summary_df = issue_matrix.sum().rename('issue_count').reset_index().rename(columns={'index': 'field'})
summary_df.to_csv(summary_csv, index=False)

# Save human-readable HTML report
html_content = styled_qc.to_html()
html_report.write_text(html_content, encoding='utf-8')

print('Saved files:')
print('-', input_csv)
print('-', issues_csv)
print('-', summary_csv)
print('-', html_report)

display(summary_df)

## Step 5 - Prepare Detection Results and High-Resolution Drawing

Now we simulate object detection output and draw it on a high-resolution engineering drawing.

### Why this matters

A QC dashboard is stronger when users can visually verify extracted objects against the source image. We will:

1. Create a synthetic high-resolution drawing.
2. Create mock detection results (`bbox + confidence + class`).
3. Overlay detections with OpenCV in color.

In [ ]:
# Create a synthetic high-resolution drawing (white canvas + thin gray guide lines)
h, w = 1800, 2400
base_img = np.full((h, w, 3), 255, dtype=np.uint8)

for x in range(0, w, 120):
    cv2.line(base_img, (x, 0), (x, h), (230, 230, 230), 1)
for y in range(0, h, 120):
    cv2.line(base_img, (0, y), (w, y), (230, 230, 230), 1)

# Draw some pseudo symbols
cv2.circle(base_img, (450, 500), 70, (60, 60, 60), 3)
cv2.rectangle(base_img, (1000, 300), (1300, 520), (60, 60, 60), 3)
cv2.line(base_img, (1500, 1000), (1900, 1300), (60, 60, 60), 4)

base_img_path = IMG_DIR / 'drawing_base.png'
cv2.imwrite(str(base_img_path), base_img)
print('Base drawing saved:', base_img_path)

# Mock detection outputs
# bbox format: [x1, y1, x2, y2]
detections = [
    {'det_id': 'D1', 'class_name': 'Valve', 'confidence': 0.94, 'bbox': [360, 410, 540, 590], 'qc_flag': 'ok'},
    {'det_id': 'D2', 'class_name': 'Pump',  'confidence': 0.62, 'bbox': [980, 260, 1320, 560], 'qc_flag': 'review'},
    {'det_id': 'D3', 'class_name': 'Pipe',  'confidence': 0.87, 'bbox': [1480, 980, 1920, 1320], 'qc_flag': 'ok'},
]

detection_df = pd.DataFrame(detections)
detection_df

In [ ]:
def draw_detection_overlay(image: np.ndarray, row: pd.Series) -> np.ndarray:
    img = image.copy()
    x1, y1, x2, y2 = row['bbox']

    # Color policy: green for ok, orange for review
    color = (0, 180, 0) if row['qc_flag'] == 'ok' else (0, 140, 255)

    cv2.rectangle(img, (x1, y1), (x2, y2), color, 5)
    label = f"{row['det_id']} | {row['class_name']} | conf={row['confidence']:.2f}"
    cv2.putText(img, label, (x1, max(30, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3, cv2.LINE_AA)
    return img

# Quick preview of all detections overlayed
overlay_all = base_img.copy()
for _, det_row in detection_df.iterrows():
    overlay_all = draw_detection_overlay(overlay_all, det_row)

plt.figure(figsize=(14, 9))
plt.imshow(cv2.cvtColor(overlay_all, cv2.COLOR_BGR2RGB))
plt.title('Dynamic Image Overlay - All Detections')
plt.axis('off')
plt.show()

## Step 6 - Create a Simple UI to Visualize Detection Results

This UI lets learners:

- Pick one detection ID from a dropdown.
- See only that bounding box overlaid on the source image.
- Review metadata (`class`, `confidence`, `QC flag`).

This pattern can later be converted into a full web dashboard (e.g., Streamlit, Dash, Gradio).

In [ ]:
det_dropdown = widgets.Dropdown(
    options=detection_df['det_id'].tolist(),
    description='Detection:',
    style={'description_width': 'initial'}
)
ui_output = widgets.Output()

def render_single_detection(change=None):
    with ui_output:
        clear_output(wait=True)
        det_id = det_dropdown.value
        row = detection_df[detection_df['det_id'] == det_id].iloc[0]
        img = draw_detection_overlay(base_img, row)

        fig, ax = plt.subplots(figsize=(12, 8))
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Detection {row['det_id']} | class={row['class_name']} | conf={row['confidence']:.2f} | qc={row['qc_flag']}")
        ax.axis('off')
        plt.show()

        display(pd.DataFrame([row]))

det_dropdown.observe(render_single_detection, names='value')
display(det_dropdown, ui_output)
render_single_detection()

## Step 7 - Human-in-the-Loop (HITL) Manual Correction System

AI extraction should not be the final authority for engineering QC.

In this section, users can:

- Select an item to review.
- Edit incorrect values manually.
- Save corrections into a separate reviewed dataset.

This creates a practical feedback loop for model retraining and process improvement.

In [ ]:
review_df = qc_df.copy()
review_df['review_status'] = np.where(issue_matrix.any(axis=1), 'needs_review', 'auto_pass')
review_df['review_comment'] = ''

item_selector = widgets.Dropdown(
    options=review_df['item_id'].tolist(),
    description='Item to review:',
    style={'description_width': 'initial'}
)

pressure_input = widgets.FloatText(description='pressure_bar')
temp_input = widgets.FloatText(description='temperature_c')
conf_input = widgets.FloatText(description='confidence')
drawing_input = widgets.Text(description='drawing_no')
comment_input = widgets.Text(description='comment')

save_btn = widgets.Button(description='Save Correction', button_style='success')
reload_btn = widgets.Button(description='Load Selected Item')
review_out = widgets.Output()

def load_item(_=None):
    item_id = item_selector.value
    row = review_df[review_df['item_id'] == item_id].iloc[0]
    pressure_input.value = float(row['pressure_bar'])
    temp_input.value = float(row['temperature_c'])
    conf_input.value = float(row['confidence'])
    drawing_input.value = str(row['drawing_no'])
    comment_input.value = str(row['review_comment'])

    with review_out:
        clear_output(wait=True)
        display(HTML(f"<b>Loaded item:</b> {item_id} | <b>Current status:</b> {row['review_status']}"))

def save_item(_):
    item_id = item_selector.value
    idx = review_df.index[review_df['item_id'] == item_id][0]

    review_df.at[idx, 'pressure_bar'] = pressure_input.value
    review_df.at[idx, 'temperature_c'] = temp_input.value
    review_df.at[idx, 'confidence'] = conf_input.value
    review_df.at[idx, 'drawing_no'] = drawing_input.value.strip()
    review_df.at[idx, 'review_comment'] = comment_input.value.strip()

    # Re-evaluate QC for this record after manual edit
    new_issue = build_issue_matrix(review_df.loc[[idx], ['item_id','component_type','pressure_bar','temperature_c','confidence','drawing_no']])
    review_df.at[idx, 'review_status'] = 'corrected_pass' if not new_issue.any(axis=1).iloc[0] else 'still_invalid'

    with review_out:
        clear_output(wait=True)
        display(HTML(f"<span style='color:green;'><b>Saved:</b> {item_id}</span>"))
        display(review_df.loc[[idx]])

reload_btn.on_click(load_item)
save_btn.on_click(save_item)
item_selector.observe(load_item, names='value')

form = widgets.VBox([
    item_selector,
    widgets.HBox([reload_btn, save_btn]),
    pressure_input,
    temp_input,
    conf_input,
    drawing_input,
    comment_input,
    review_out
])

display(form)
load_item()

## Step 8 - Export Corrected Results and Review Audit

After human correction, we should persist outputs for traceability and downstream systems.

In this step, you will:

1. Export the corrected table to `qc_reviewed_data.csv`.
2. Export an audit artifact to `qc_review_audit.json`.
3. Confirm review status distribution (e.g., `corrected_pass`, `still_invalid`, `auto_pass`).

### Why this step is important

A QC dashboard is not only about visualization. It must also provide an auditable trail so teams can:

- track what was changed manually,
- measure model quality over time,
- build retraining datasets from corrected labels.


In [ ]:
# Step 8 - Export reviewed data and audit log
reviewed_csv = REPORT_DIR / 'qc_reviewed_data.csv'
audit_json = REPORT_DIR / 'qc_review_audit.json'

review_df.to_csv(reviewed_csv, index=False)

# Simple audit artifact for downstream systems
payload = {
    'total_records': int(len(review_df)),
    'status_counts': review_df['review_status'].value_counts().to_dict(),
    'records': review_df.to_dict(orient='records')
}
audit_json.write_text(json.dumps(payload, indent=2), encoding='utf-8')

print('Reviewed dataset saved:', reviewed_csv)
print('Audit JSON saved      :', audit_json)

display(review_df)

## Step 9 - Suggested Exercises for Learners

To deepen understanding, try these tasks:

1. Add new QC rules (e.g., component-specific pressure limits).
2. Add severity levels (`warning`, `critical`) and color mapping.
3. Track who reviewed each record and when (`reviewer`, `timestamp`).
4. Build a trend chart of daily QC issue counts.
5. Replace synthetic data with real model output files.

---

## Wrap-up

You now have a complete prototype pipeline for a QC dashboard:

- Automated consistency checks with red highlighting.
- Report generation for machine and human consumption.
- Visual verification using OpenCV dynamic overlays.
- Human-in-the-loop correction with persistent reviewed outputs.

This is the practical foundation for production QC systems in CAD/engineering extraction workflows.

## Extended Practice Track (Long-Form Course)

Great QC engineers improve by repeating many realistic tasks.

From this section onward, you will work through **many additional exercises** to extend course duration and hands-on depth.

Each new step includes:

- A clear objective
- Runnable Python code
- Expected outputs to verify
- A mini-task for learners

> Recommended: run every step and save outputs into `qc_dashboard_outputs/reports` for your class portfolio.

In [ ]:
severity_weights = {
    'pressure_bar': 3,      # high impact
    'temperature_c': 3,     # high impact
    'confidence': 2,        # medium impact
    'drawing_no': 1         # low-medium impact
}

priority_df = qc_df.copy()
priority_df['issue_count'] = issue_matrix.sum(axis=1)
priority_df['priority_score'] = 0

for col, w in severity_weights.items():
    priority_df['priority_score'] += issue_matrix[col].astype(int) * w

priority_df['review_priority'] = pd.cut(
    priority_df['priority_score'],
    bins=[-1, 0, 2, 5, 99],
    labels=['pass', 'low', 'medium', 'high']
)

priority_df = priority_df.sort_values(['priority_score', 'issue_count'], ascending=False)
priority_df

## Step 11 - Generate a Larger Synthetic Dataset (Batch QC Simulation)

### Objective

Simulate a realistic batch (hundreds of records) to stress-test your QC logic.

### Learner Task

- Increase sample size from `500` to `2000`.
- Observe processing speed and issue distribution.

In [ ]:
# Step 11 - Build synthetic batch and run build_issue_matrix() on it

np.random.seed(42)

n = 500
components = ['Valve', 'Pump', 'Tank', 'Filter', 'Motor', 'Pipe']

batch_df = pd.DataFrame({
    'item_id': [f'B-{i:04d}' for i in range(n)],
    'component_type': np.random.choice(components, size=n),
    'pressure_bar': np.random.normal(loc=5, scale=8, size=n),
    'temperature_c': np.random.normal(loc=80, scale=120, size=n),
    'confidence': np.random.normal(loc=0.85, scale=0.2, size=n),
    'drawing_no': np.random.choice(
        ['DWG-100', 'DWG-200', 'DWG-300', ''],
        size=n,
        p=[0.32, 0.32, 0.30, 0.06],
    ),
})

batch_issues = build_issue_matrix(batch_df)
batch_summary = (
    batch_issues.sum()
    .rename('issue_count')
    .reset_index()
    .rename(columns={'index': 'field'})
)
batch_df['has_issue'] = batch_issues.any(axis=1)

print('Total records:', len(batch_df))
print('Rows with issues:', int(batch_df['has_issue'].sum()))
display(batch_summary)
batch_df.head()


## Step 12 - QC KPI Dashboard (Issue Rate, Field Breakdown, Priority)

### Objective

Build compact KPI charts for quick quality monitoring.

### Learner Task

- Identify the field with highest issue rate.
- Explain why that field might be unstable in OCR/AI extraction.

In [ ]:
issue_rate = batch_issues.mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Chart 1: Issue rate by field
axes[0].bar(issue_rate.index, issue_rate.values, color='#d9534f')
axes[0].set_title('Issue Rate by Field')
axes[0].set_ylabel('Rate')
axes[0].tick_params(axis='x', rotation=30)

# Chart 2: Pass vs Issue
status_counts = batch_df['has_issue'].map({True: 'Issue', False: 'Pass'}).value_counts()
axes[1].pie(status_counts.values, labels=status_counts.index, autopct='%.1f%%', colors=['#f0ad4e','#5cb85c'])
axes[1].set_title('Overall QC Status')

# Chart 3: Component distribution for problematic records
problem_components = batch_df.loc[batch_df['has_issue'], 'component_type'].value_counts()
axes[2].bar(problem_components.index, problem_components.values, color='#5bc0de')
axes[2].set_title('Issue Count by Component')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
queue_df = batch_df.copy()
queue_df = queue_df.assign(issue_count=batch_issues.sum(axis=1))
queue_df['priority_score'] = (
    batch_issues['pressure_bar'].astype(int) * 3 +
    batch_issues['temperature_c'].astype(int) * 3 +
    batch_issues['confidence'].astype(int) * 2 +
    batch_issues['drawing_no'].astype(int) * 1
)

queue_df['sla_bucket'] = pd.cut(
    queue_df['priority_score'],
    bins=[-1, 0, 2, 4, 99],
    labels=['none', 'this_week', 'today', 'urgent']
)

review_queue = queue_df[queue_df['priority_score'] > 0].sort_values('priority_score', ascending=False)

print('Total queue size:', len(review_queue))
display(review_queue[['item_id','component_type','issue_count','priority_score','sla_bucket']].head(15))
print('\nSLA distribution:')
display(review_queue['sla_bucket'].value_counts())

## Step 14 - Simulate Multi-Reviewer Assignment

### Objective

Assign review tasks to multiple reviewers and balance workload.

### Learner Task

- Add more reviewer names.
- Compare load balancing strategies (`round-robin`, `priority-aware`).

In [ ]:
reviewers = ["Alice", "Bob", "Charlie", "Diana"]
assign_df = review_queue.copy().reset_index(drop=True)
assign_df['assigned_to'] = [reviewers[i % len(reviewers)] for i in range(len(assign_df))]

workload = assign_df.groupby('assigned_to').agg(
    tasks=('item_id', 'count'),
    total_priority=('priority_score', 'sum')
).sort_values("tasks", ascending=False)

display(assign_df[['item_id','priority_score','sla_bucket','assigned_to']].head(20))
print('Reviewer workload summary')
display(workload)


## Step 15 - Track Human Corrections and Compute Improvement Metrics

### Objective

Measure how much human review improves data quality.

### Learner Task

- Change correction strategy and recompute improvement.
- Explain which fields require model retraining priority.

In [ ]:
# Simulate reviewer corrections on problematic rows
corrected_df = batch_df.copy()
needs_fix = batch_issues.any(axis=1)

# Correction policy (for teaching): clip invalid ranges and fill missing drawing_no
corrected_df.loc[:, 'pressure_bar'] = corrected_df['pressure_bar'].clip(lower=0, upper=300)
corrected_df.loc[:, 'temperature_c'] = corrected_df['temperature_c'].clip(lower=-40, upper=350)
corrected_df.loc[:, 'confidence'] = corrected_df['confidence'].clip(lower=0, upper=1)
corrected_df.loc[corrected_df['drawing_no'].astype(str).str.strip().eq(''), 'drawing_no'] = 'DWG-MANUAL'

after_issues = build_issue_matrix(corrected_df)

metrics = pd.DataFrame({
    'before_issue_count': batch_issues.sum(),
    'after_issue_count': after_issues.sum()
})
metrics['improvement'] = metrics['before_issue_count'] - metrics['after_issue_count']
metrics['improvement_rate'] = np.where(metrics['before_issue_count'] > 0,
                                      metrics['improvement'] / metrics['before_issue_count'], 0.0)

print('Rows originally needing review:', int(needs_fix.sum()))
display(metrics)

In [ ]:
compare_cols = ['pressure_bar', 'temperature_c', 'confidence', 'drawing_no']
retrain_df = batch_df[['item_id','component_type'] + compare_cols].copy()

for c in compare_cols:
    retrain_df[f'ai_{c}'] = batch_df[c]
    retrain_df[f'human_{c}'] = corrected_df[c]
    retrain_df[f'changed_{c}'] = retrain_df[f'ai_{c}'] != retrain_df[f'human_{c}']

retrain_df['changed_any'] = retrain_df[[f'changed_{c}' for c in compare_cols]].any(axis=1)
retrain_samples = retrain_df[retrain_df['changed_any']].copy()

retrain_path = REPORT_DIR / 'qc_retraining_samples.csv'
retrain_samples.to_csv(retrain_path, index=False)

print('Retraining sample count:', len(retrain_samples))
print('Saved:', retrain_path)
retrain_samples.head(10)

## Step 17 - Export Batch Overlay Images for Reviewer Packs

### Objective

Generate many overlay images automatically for offline reviewer packs.

### Learner Task

- Increase number of generated images.
- Add filename convention with date + reviewer.

In [ ]:
overlay_batch_dir = IMG_DIR / "batch_overlays"
overlay_batch_dir.mkdir(exist_ok=True, parents=True)

np.random.seed(123)
for i in range(20):
    img = base_img.copy()
    x1 = np.random.randint(100, 2000)
    y1 = np.random.randint(100, 1400)
    x2 = min(x1 + np.random.randint(120, 350), w - 1)
    y2 = min(y1 + np.random.randint(120, 350), h - 1)
    conf = np.random.uniform(0.5, 0.98)
    qc_flag = 'ok' if conf >= 0.75 else 'review'
    color = (0, 180, 0) if qc_flag == 'ok' else (0, 140, 255)
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 5)
    cv2.putText(img, f'AutoDet-{i:02d} conf={conf:.2f} {qc_flag}', (x1, max(40, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3, cv2.LINE_AA)
    out_path = overlay_batch_dir / f'overlay_{i:02d}.png'
    cv2.imwrite(str(out_path), img)

print('Generated overlay images in:', overlay_batch_dir)


## Step 18 - Add Basic Unit Tests for QC Functions

### Objective

Ensure your QC functions stay reliable when notebook logic evolves.

### Learner Task

- Add one more test case with edge values.
- Intentionally break a rule and observe failed assertion.

In [ ]:
test_df = pd.DataFrame([
    {'item_id':'X1', 'component_type':'Valve', 'pressure_bar':10, 'temperature_c':50, 'confidence':0.9, 'drawing_no':'D1'},
    {'item_id':'X2', 'component_type':'Pump',  'pressure_bar':-5, 'temperature_c':50, 'confidence':0.9, 'drawing_no':'D2'},
    {'item_id':'X3', 'component_type':'Tank',  'pressure_bar':10, 'temperature_c':500, 'confidence':0.9, 'drawing_no':'D3'},
    {'item_id':'X4', 'component_type':'Pipe',  'pressure_bar':10, 'temperature_c':50, 'confidence':1.5, 'drawing_no':'D4'},
    {'item_id':'X5', 'component_type':'Motor', 'pressure_bar':10, 'temperature_c':50, 'confidence':0.9, 'drawing_no':''},
])

res = build_issue_matrix(test_df)

assert res.loc[0].sum() == 0, 'X1 should pass all checks'
assert res.loc[1, 'pressure_bar'] == True, 'X2 pressure check should fail'
assert res.loc[2, 'temperature_c'] == True, 'X3 temperature check should fail'
assert res.loc[3, 'confidence'] == True, 'X4 confidence check should fail'
assert res.loc[4, 'drawing_no'] == True, 'X5 drawing_no check should fail'

print('All QC unit tests passed.')

## Step 19 - Build a Lightweight Monthly Monitoring Simulation

### Objective

Simulate QC metrics over multiple months to teach trend analysis.

### Learner Task

- Change month count from 6 to 12.
- Plot additional metric: corrected pass rate.

In [ ]:
months = pd.period_range("2026-01", periods=6, freq="M").astype(str)
trend = pd.DataFrame({
    'month': months,
    'records_processed': np.random.randint(350, 700, size=len(months)),
    'issue_rate': np.round(np.random.uniform(0.12, 0.35, size=len(months)), 3),
    'review_sla_hit_rate': np.round(np.random.uniform(0.78, 0.96, size=len(months)), 3)
})

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(trend['month'], trend['issue_rate'], marker='o', label='Issue Rate', color='#d9534f')
ax1.plot(trend['month'], trend['review_sla_hit_rate'], marker='o', label='SLA Hit Rate', color='#5cb85c')
ax1.set_ylim(0, 1)
ax1.set_ylabel('Rate')
ax1.set_title('Monthly QC Trend Simulation')
ax1.legend(loc='best')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

trend


## Step 20 - Capstone Tasks (Large Practice Set)

Complete as many tasks as possible:

1. Add component-specific QC rules (different limits by component type).
2. Add reviewer identity, timestamp, and approval signature columns.
3. Build a retry queue for `still_invalid` records.
4. Add confidence calibration buckets (0.0-0.5, 0.5-0.7, 0.7-0.9, 0.9-1.0).
5. Create a final executive report with KPIs + top root causes.
6. Save all artifacts to a dated folder (e.g., `reports/2026-04-30/`).
7. Add a confusion-style table comparing AI flags vs human decisions.
8. Design a policy for auto-approval vs manual review threshold.
9. Integrate this notebook with your real YOLO detection outputs.
10. Present a 10-minute QC pipeline demo to your class.

---

## Final Course Note

You now have a long-form practice notebook with many tasks to support extended training sessions.

If you want, the next upgrade can be:

- a Streamlit dashboard version,
- or a production-ready modular Python package (`/src`, `/tests`, `/configs`) built from this notebook.

## Additional Worked Examples (Extended Practice)

Below are **many short examples** you can run independently. Each illustrates one QC pattern common in production dashboards.

**Tip:** Duplicate any cell and tweak parameters to create your own homework drills.


### Example 21 - Component-Specific Numeric Limits

Different equipment types often need different engineering ranges. This pattern applies row-wise limits using a lookup table.


In [ ]:
limits = {
    'Valve': {'p_min': 0, 'p_max': 50},
    'Pump': {'p_min': 0, 'p_max': 200},
    'Tank': {'p_min': 0, 'p_max': 20},
    'Filter': {'p_min': 0, 'p_max': 30},
    'Motor': {'p_min': 0, 'p_max': 10},
    'Pipe': {'p_min': 0, 'p_max': 300},
}

ex21 = batch_df.head(50).copy()


def pressure_out_of_component_range(row):
    spec = limits.get(row['component_type'], {'p_min': 0, 'p_max': 300})
    return not spec['p_min'] <= row['pressure_bar'] <= spec['p_max']


ex21['pressure_component_mismatch'] = ex21.apply(pressure_out_of_component_range, axis=1)
print('Rows with component-specific pressure issues:', int(ex21['pressure_component_mismatch'].sum()))
ex21[ex21['pressure_component_mismatch']].head(10)


### Example 22 - Drawing Number Format (Regex QC)

Flag drawing references that do not match an expected pattern (for example a fixed prefix + digits).


In [ ]:
import re

drawing_pattern = re.compile(r"^DWG-\d{3,5}$", re.I)
ex22 = batch_df[["item_id", "drawing_no"]].copy()
ex22['drawing_format_ok'] = (
    ex22['drawing_no']
    .astype(str)
    .str.strip()
    .map(lambda s: bool(drawing_pattern.match(s)) if s else False)
)
ex22['drawing_missing_or_bad'] = ~ex22['drawing_format_ok']
print('Bad/missing drawing format (demo rule):', int(ex22['drawing_missing_or_bad'].sum()))
ex22[ex22['drawing_missing_or_bad']].head(12)


### Example 23 - Duplicate Keys

Duplicates cause double-counting in BOM or hazard analysis. Flag repeated `item_id` values.


In [ ]:
dup_ids = batch_df["item_id"][batch_df["item_id"].duplicated(keep=False)]
print("Duplicate item_id count in synthetic batch:", len(dup_ids))
dup_demo = pd.concat(
    [batch_df.iloc[[0, 1]], batch_df.iloc[[0, 1]].assign(item_id=lambda d: d['item_id'] + '-COPY')],
    ignore_index=True,
)
dup_demo['_dup'] = dup_demo['item_id'].duplicated(keep=False)
dup_demo


### Example 24 - Cross-Field Consistency (Tank + High Pressure)

Teaching scenario: if component is `Tank`, pressure above 15 bar is suspicious — replace thresholds with your plant data.


In [ ]:
ex24 = batch_df.copy()
ex24['tank_high_pressure'] = (ex24['component_type'] == 'Tank') & (ex24['pressure_bar'] > 15)
print('Tank high-pressure flags:', int(ex24['tank_high_pressure'].sum()))
ex24.loc[ex24['tank_high_pressure'], ['item_id', 'pressure_bar', 'temperature_c']].head(10)


### Example 25 - Text Normalization

Leading/trailing spaces and odd whitespace often break joins. Normalize before QC.


In [ ]:
messy = pd.DataFrame({"drawing_no": ["  DWG-100 ", "DWG-200\t", "  ", "dwg-300"]})
messy['drawing_clean'] = (
    messy['drawing_no'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip().str.upper()
)
messy['was_empty'] = messy['drawing_clean'].eq('')
messy


### Example 26 - Statistical Outliers (IQR) per Component Group

Outliers are not always errors, but they are useful manual-review candidates.


In [ ]:
def iqr_flags(series: pd.Series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    return (series < lo) | (series > hi)


ex26 = batch_df.copy()
ex26['pressure_outlier_iqr'] = ex26.groupby('component_type')['pressure_bar'].transform(lambda s: iqr_flags(s))
print('IQR pressure outliers:', int(ex26['pressure_outlier_iqr'].sum()))
ex26.loc[ex26['pressure_outlier_iqr'], ['item_id', 'component_type', 'pressure_bar']].head(12)


### Example 27 - Confidence Buckets vs Issue Rate

Check whether low-confidence predictions correlate with rule violations.


In [ ]:
ex27 = batch_df.copy()
ex27['conf_bucket'] = pd.cut(
    ex27['confidence'], bins=[0, 0.5, 0.7, 0.9, 1.0], labels=['0-0.5', '0.5-0.7', '0.7-0.9', '0.9-1.0']
)
ex27['has_issue'] = batch_issues.any(axis=1).values
bucket_stats = ex27.groupby('conf_bucket', observed=True)['has_issue'].agg(['count', 'mean']).rename(
    columns={'mean': 'issue_rate'}
)
bucket_stats


### Example 28 - Stratified QC Sampling

Sample a fixed number per component so rare symbols still appear in manual review.


In [ ]:
rng = np.random.default_rng(7)
sample_per_comp = 8
parts = []
for _, grp in batch_df.groupby('component_type'):
    take = min(sample_per_comp, len(grp))
    parts.append(grp.sample(take, random_state=rng))
audit_sample = pd.concat(parts, ignore_index=True)
print('Audit sample size:', len(audit_sample))
audit_sample['component_type'].value_counts()


### Example 29 - Simple Drift Check Between Two Synthetic Batches

Compare summary statistics when pipelines or scanners change.


In [ ]:
rng = np.random.default_rng(1)
batch_a = rng.normal(5, 2, size=300)
batch_b = rng.normal(7.5, 2.2, size=300)
drift_report = pd.DataFrame(
    {
        'batch': ['A', 'B'],
        'mean_pressure': [batch_a.mean(), batch_b.mean()],
        'std_pressure': [batch_a.std(), batch_b.std()],
    }
)
drift_report


### Example 30 - Detection List → Class-Colored Overlay + Legend

Shows how to consume structured detection output (similar to YOLO post-processing) for QC visualization.


In [ ]:
CLASS_COLORS = {"Valve": (0, 165, 255), "Pump": (255, 0, 0), "Pipe": (0, 255, 0), "Tank": (255, 0, 255)}
detections_json = [
    {"cls": "Valve", "bbox": [300, 400, 480, 560], "score": 0.91},
    {"cls": "Pump", "bbox": [900, 280, 1280, 560], "score": 0.63},
    {"cls": "Pipe", "bbox": [1450, 980, 1950, 1340], "score": 0.88},
]
vis = base_img.copy()
y0 = 40
for cls_name, bgr in CLASS_COLORS.items():
    cv2.putText(vis, cls_name, (40, y0), cv2.FONT_HERSHEY_SIMPLEX, 1.0, bgr, 2, cv2.LINE_AA)
    y0 += 40
for d in detections_json:
    x1, y1, x2, y2 = d["bbox"]
    col = CLASS_COLORS.get(d["cls"], (200, 200, 200))
    cv2.rectangle(vis, (x1, y1), (x2, y2), col, 4)
    label = "%s %.2f" % (d["cls"], d["score"])
    cv2.putText(vis, label, (x1, max(35, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, col, 2, cv2.LINE_AA)
legend_path = IMG_DIR / "overlay_class_legend_demo.png"
cv2.imwrite(str(legend_path), vis)
print('Saved:', legend_path)
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()


### Example 31 - Append-Only QC Event Log (JSON Lines)

Each QC run or export can append one JSON object per line for downstream analytics.


In [ ]:
from datetime import datetime, timezone

log_path = REPORT_DIR / "qc_events.jsonl"
events = [
    {
        "ts": datetime.now(timezone.utc).isoformat(),
        "event": "qc_run",
        "records": len(batch_df),
        "issues": int(batch_issues.any(axis=1).sum()),
    },
    {
        "ts": datetime.now(timezone.utc).isoformat(),
        "event": "sample_export",
        "path": str(log_path),
    },
]
with open(log_path, "w", encoding="utf-8") as f:
    for e in events:
        f.write(json.dumps(e) + "\n")
print('Wrote', len(events), 'events to', log_path)
Path(log_path).read_text(encoding='utf-8').strip().splitlines()[-2:]


### Example 32 - Simulated Agreement Table (AI Flag vs Human Decision)

In real dashboards you compare model auto-QC against reviewer disposition.


In [ ]:
rng = np.random.default_rng(99)
n_sim = 200
ai_flag = rng.random(n_sim) < 0.22
human_flag = np.where(ai_flag, rng.random(n_sim) < 0.35, rng.random(n_sim) < 0.08)
pd.crosstab(pd.Series(ai_flag, name='AI_issue'), pd.Series(human_flag, name='Human_issue'), margins=True)


### Example 33 - Export Machine-Readable QC Rule Config

Store thresholds in JSON so engineers can change limits without editing Python.


In [ ]:
rule_config = {
    "pressure_bar": {"min": 0, "max": 300},
    "temperature_c": {"min": -40, "max": 350},
    "confidence": {"min": 0, "max": 1},
    "drawing_no": {"non_empty": True},
}
rules_path = REPORT_DIR / "qc_rule_config.json"
rules_path.write_text(json.dumps(rule_config, indent=2), encoding='utf-8')
print('Saved:', rules_path)


### Example 34 - Multi-Rule Summary per Row (Explainability)

Concatenate human-readable reasons — helpful in reviewer UI tooltips.


In [ ]:
named = issue_matrix.copy()
for col in named.columns:
    named[col] = np.where(named[col], col, '')
qc_df_demo = qc_df.copy()
qc_df_demo['qc_reasons'] = named.apply(lambda r: '; '.join([x for x in r if x]), axis=1)
qc_df_demo['qc_reasons'] = qc_df_demo['qc_reasons'].replace('', '(none)')
qc_df_demo[['item_id', 'qc_reasons']]


### Example 35 - Rolling Issue Rate (Time-Series Style)

If each row has a timestamp, you can plot rolling QC burden — here we simulate ingest timestamps.


In [ ]:
rng = np.random.default_rng(5)
ts_df = batch_df.copy()
ts_df['ingested_at'] = pd.date_range('2026-04-01', periods=len(ts_df), freq='h')
ts_df['issue'] = batch_issues.any(axis=1).astype(int).values
ts_df = ts_df.sort_values('ingested_at')
ts_df['rolling_issue_rate'] = ts_df['issue'].rolling(48, min_periods=10).mean()
plt.figure(figsize=(11, 4))
plt.plot(ts_df['ingested_at'], ts_df['rolling_issue_rate'], color='#d9534f')
plt.title('Rolling 48h issue rate (simulated ingest timestamps)')
plt.ylabel('Rate')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


### Example 36 - Threshold Sweeps for Auto-Approval Policy

Explore how many rows would auto-pass if you trust only high-confidence extractions.


In [ ]:
thresholds = np.linspace(0.5, 0.99, 11)
rows = []
has_issue = batch_issues.any(axis=1).values
conf = batch_df['confidence'].values
for t in thresholds:
    auto = conf >= t
    rows.append(
        {
            'threshold': round(float(t), 3),
            'auto_fraction': float(auto.mean()),
            'issue_rate_if_auto': float(has_issue[auto].mean()) if auto.any() else float('nan'),
        }
    )
policy_df = pd.DataFrame(rows)
policy_df


### Example 37 - Bounding Box Sanity Checks (Vision QC)

Catch degenerate boxes before drawing overlays (zero area, inverted coordinates).


In [ ]:
bbox_rows = [
    {"det_id": "OK", "bbox": [10, 10, 50, 60]},
    {"det_id": "INV", "bbox": [50, 50, 40, 90]},
    {"det_id": "ZERO", "bbox": [100, 100, 100, 100]},
]
bb = pd.DataFrame(bbox_rows)


def bbox_invalid(b):
    x1, y1, x2, y2 = b
    if x2 <= x1 or y2 <= y1:
        return True
    if (x2 - x1) * (y2 - y1) < 4:
        return True
    return False


bb['bbox_bad'] = bb['bbox'].map(bbox_invalid)
bb


### Example 38 - Batch Export Styled HTML Table (Alternative Report)

Generate a second HTML artifact focused on the batch dataframe head.


In [ ]:
batch_preview_html = REPORT_DIR / "batch_preview_report.html"
preview = batch_df.head(30).copy()
preview_issues = build_issue_matrix(preview)
styler = preview.style.apply(lambda _: style_invalid_values(preview, preview_issues), axis=None)
batch_preview_html.write_text(styler.to_html(), encoding='utf-8')
print('Saved:', batch_preview_html)


### Example 39 - Pareto Chart of QC Issues (Field Contributions)

Identify which fields contribute most to total flags.


In [ ]:
counts = batch_issues.sum().sort_values(ascending=False)
cum_pct = counts.cumsum() / counts.sum()
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.bar(counts.index, counts.values, color='#5bc0de')
ax1.set_ylabel('Issue count')
ax2 = ax1.twinx()
ax2.plot(counts.index, cum_pct.values, color='#d9534f', marker='o')
ax2.set_ylim(0, 1)
ax2.set_ylabel('Cumulative fraction')
plt.title('Pareto of QC issues by field')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


### Example 40 - Practice Checklist (Self-Study)

Repeat each mini-drill with new random seeds:

1. Tighten regex for drawing numbers.
2. Add `revision` column QC (`REV-A`, `REV-B`).
3. Implement severity matrix with customer SLA.
4. Connect `detections_json` to your real inference JSON.
5. Store reviewer edits with UUID primary keys.


In [ ]:
# No code — duplicate earlier examples and vary parameters.
